# Importation of necessary Python Libs

## Maths or data structure-related libraries

In [5]:
import math
import numpy as np
import random
import pandas as pd
from collections import deque, namedtuple
from re import match
from typing import Tuple, List, Set, Dict
from sklearn.linear_model import BayesianRidge

## Libraries for visualization/record tracking

In [6]:
import matplotlib.pyplot as plt
import tqdm

## Libraries for loading data

In [15]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os

Mounted at /content/drive


## Libraries for machine leanring

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer

# Implementation of the VAEQL algoritm

The link to my manuscript: <a>https://www.overleaf.com/project/67cb575babefcc1067d01469</a>

## Defining the miscelaneous methods

### Defining the training method depending on the hardware device

In [8]:
def training_func():
    device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu" # CUDA doesn't work with the AMD GPUs of MacBook M1
    if device == "cuda:0":
        print("Training on the GPU")
    else:
        print("Training on the CPU")

    ENV = namedtuple('env', ('name', 'n_actions', 'encoding_dim'))

### Copy and paste the benchmark dataframe pre-processing methods from the previous experiments

In [9]:
def identify_binary_and_numerical_features(df: pd.DataFrame) -> Tuple[List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]

    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [10]:
def normalize_numerical_features(df: pd.DataFrame, num_features: List[str]) -> pd.DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[num_features] = scaler.fit_transform(df[num_features])

    return df

In [34]:
def generate_masks_for_missingness(
    input_df: pd.DataFrame,
    original_df: pd.DataFrame,
    amputed_df: pd.DataFrame,
    num_feats: List[str],
    cat_feats: List[str],
) -> Tuple[np.ndarray, np.ndarray]:

    # Check if the dataframes match in shape
    if not original_df.shape == input_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(input_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int)

    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed

    # Generate the missingness map for one-hot-encoded categorical features
    for i, feat in enumerate(cat_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

## Defining the Variational Autoencoder part of the VAEQL algorithm

### The VAE module

In [12]:
class MaskedVAE(nn.Module):
    def __init__(
        self,
        num_features: int,
        latent_dim: int | None = None,
        hidden_layer_sizes: tuple[int, ...] = (128, 64)
    ):
        super().__init__()
        # Determine latent dimension (50% reduction if not specified)
        self.num_features = num_features
        self.latent_dim = latent_dim or (num_features // 2)

        # Build encoder: fully connected layers from num_features → latent_dim
        encoder_layers = []
        in_dim = num_features
        for hidden_dim in hidden_layer_sizes:
            encoder_layers.append(nn.Linear(in_dim, hidden_dim))
            encoder_layers.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        self.encoder = nn.Sequential(*encoder_layers)
        # Map to latent parameters
        self.fc1 = nn.Linear(in_dim, self.latent_dim)  # μ
        self.fc2 = nn.Linear(in_dim, self.latent_dim)  # log(σ²)

        # Build decoder: latent_dim → num_features
        decoder_layers = []
        in_dim = self.latent_dim
        for hidden_dim in reversed(hidden_layer_sizes):
            decoder_layers.append(nn.Linear(in_dim, hidden_dim))
            decoder_layers.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        decoder_layers.append(nn.Linear(in_dim, num_features))
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(
        self,
        features: torch.Tensor,
        mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            features: (batch_size, num_features) tensor of amputed+normalized data
            mask:     (batch_size, num_features) numpy-based mask array passed as tensor
                      values in {0,1,2}; only used in loss calculation
                      (0=originally observed and not amputed, 1=originally missing, 2=amputed)
        Returns:
            reconstructed: (batch_size, num_features), i.e., the latent representation
            mu:            (batch_size, latent_dim)
            logvar:        (batch_size, latent_dim)
        """
        # Encode
        hidden = self.encoder(features)
        mu = self.fc1(hidden)
        logvar = self.fc2(hidden)

        # Reparameterization trick
        std = torch.exp(logvar * 0.5)
        eps = torch.randn_like(std)
        z = mu + eps * std

        # Decode
        reconstructed = torch.sigmoid(self.decoder(z))
        return reconstructed, mu, logvar

### The loss function

In [13]:
def masked_vae_loss(
    reconstructed: torch.Tensor,
    original: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    mask: torch.Tensor,
    beta: float = 1.0,
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Compute masked VAE loss: reconstruction only over entries with mask==0.
    """
    # Elementwise binary cross-entropy loss
    bce = F.binary_cross_entropy(
        reconstructed, original, reduction='none'
    )
    # Only originally observed entries contribute
    observed = (mask == 0).float()
    masked_bce = bce * observed
    reconstruction_loss = masked_bce.sum() / (observed.sum() + eps)

    # KL divergence between q(z|X) and p(z) = N(0,I)
    kl_div = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()

    return reconstruction_loss + beta * kl_div

### The VAE training method

In [61]:
def train_masked_vae(
    data_df: pd.DataFrame,
    mask_array: np.ndarray,
    hidden_layer_sizes: tuple[int, ...] = (128, 64),
    latent_dim: int | None = None,
    batch_size: int = 32,
    num_folds: int = 5,
    max_epochs: int = 1000,
    learning_rate: float = 1e-3,
    beta: float = 1.0,
) -> list[float]:
    """
    Train the MaskedVAE on a pandas DataFrame with a numpy mask.

    Args:
        data_df:    pandas.DataFrame of shape (n_samples, num_features)
        mask_array: numpy.ndarray of same shape, values {0,1,2}
        ...         other hyperparameters
    Returns:
        List of final validation losses for each fold.
    """
    # Validate shapes
    if data_df.shape != mask_array.shape:
        raise ValueError(
            f"Input data shape {data_df.shape} must match mask shape {mask_array.shape}"
        )

    # define the training device
    device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu" # CUDA doesn't work with the AMD GPUs of MacBook M1
    print(f"Training on {device} ...")

    # Convert inputs to tensors
    features = torch.from_numpy(data_df.values.astype(np.float32))
    mask = torch.from_numpy(mask_array.astype(np.int64))

    num_samples, num_features = features.shape
    latent_dim = latent_dim or (num_features // 2)

    # Prepare cross-validation
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_losses: list[float] = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(features), start=1):
        # Initialize model and optimizer
        model = MaskedVAE(
            num_features=num_features,
            latent_dim=latent_dim,
            hidden_layer_sizes=hidden_layer_sizes
        ).to(device)
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

        # Create DataLoaders
        train_ds = TensorDataset(
            features[train_idx].to(device),
            mask[train_idx].to(device)
        )
        val_ds = TensorDataset(
            features[val_idx].to(device),
            mask[val_idx].to(device)
        )
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False
        )

        val_losses: list[float] = []
        # Training loop with early stopping
        for epoch in range(1, max_epochs + 1):
            model.train()
            for batch_features, batch_mask in train_loader:
                optimizer.zero_grad()
                recon, mu, logvar = model(batch_features, batch_mask)
                loss = masked_vae_loss(
                    reconstructed=recon,
                    original=batch_features,
                    mu=mu,
                    logvar=logvar,
                    mask=batch_mask,
                    beta=beta
                )
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for batch_features, batch_mask in val_loader:
                    recon, mu, logvar = model(batch_features, batch_mask)
                    batch_loss = masked_vae_loss(
                        reconstructed=recon,
                        original=batch_features,
                        mu=mu,
                        logvar=logvar,
                        mask=batch_mask,
                        beta=beta
                    )
                    total_val_loss += batch_loss.item() * batch_features.size(0)
            avg_val_loss = total_val_loss / len(val_idx)
            val_losses.append(avg_val_loss)

            # Early stopping on 100-epoch moving average
            if epoch >= 100:
                recent_ma = sum(val_losses[-10:]) / 100
                prev_ma   = sum(val_losses[-11:-1]) / 100
                if recent_ma > prev_ma:
                    print(f"Fold {fold}: early stopping at epoch {epoch}")
                    break

        print(f"Fold {fold} final validation loss: {avg_val_loss:.4f}")
        fold_losses.append(avg_val_loss)

    return fold_losses

### Testing on the toy training method above

In [36]:
# create the list of tuples for pairing input
# 1. Point to the directory containing your CSVs
base_dir = "/content/drive/My Drive/VAEQL_paper"
amputed_dir = "/content/drive/My Drive/VAEQL_paper/amputed_datasets"
preprossed_dir = "/content/drive/My Drive/VAEQL_paper/preprocessed_datasets"

ref_dataset_dict = {
    "HCV_data": os.path.join(preprossed_dir, "HCV_data_unimputed.csv"),
    "appendicitis_data": os.path.join(preprossed_dir, "Regensburg_Pediatric_Appendicitis_unimputed.csv"),
    "Indian_liver_patients": os.path.join(preprossed_dir, "Indian_liver_patients_unimputed.csv"),
    "heart_failure_clinical_records": os.path.join(preprossed_dir, "Heart_Failure_Clinical_Records_complete.csv"),
    "HCV_Egyptian_patients": os.path.join(preprossed_dir, "HCV_Egyptian_data_complete.csv")
}

csv_path_pairs: List[Dict[str, str]] = []
for root, _, files in os.walk(amputed_dir):
    for f in files:
        if f.endswith(".csv"):
            amputed_path = os.path.join(root, f)
            dataset_name = root.split("/")[-1]
            brr_imputed_path = os.path.join(base_dir, "BRR_imputed_datasets", dataset_name, f)
            ref_dataset = os.path.join(preprossed_dir, ref_dataset_dict[dataset_name])
            csv_path_pairs.append({"amputed": amputed_path, "input": brr_imputed_path, "ref": ref_dataset})

print(f"Found {len(csv_path_pairs)} CSV file pairs in all subfolders.")

Found 500 CSV file pairs in all subfolders.


In [37]:
csv_path_pairs[:3]

[{'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MAR_20_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MAR_20_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MNAR_10_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MNAR_10_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/heart_failure_clinical_records/MAR_15_perc_1.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/heart_failure_clinical_records/MAR_15_perc_1.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preproce

In [38]:
csv_path_pairs[-3:]

[{'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_15_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_20_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_20_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv'},
 {'amputed': '/content/drive/My Drive/VAEQL_paper/amputed_datasets/appendicitis_data/MNAR_25_perc_10.csv',
  'input': '/content/drive/My Drive/VAEQL_paper/BRR_imputed_datasets/appendicitis_data/MNAR_25_perc_10.csv',
  'ref': '/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.cs

#### 1. <code>the Regensburg Pediatric Appendicitis Dataset<code> (large number of missing values, 713 patients)

##### Load and pre-process the data

In [43]:
feat_m1, feat_c1 = identify_binary_and_numerical_features(pd.read_csv(csv_path_pairs[-2]["ref"]))

In [44]:
test_apd_mask_num, test_apd_mask_cat = generate_masks_for_missingness(
    pd.read_csv(csv_path_pairs[-2]["input"]),
    pd.read_csv(csv_path_pairs[-2]["ref"]),
    pd.read_csv(csv_path_pairs[-2]["amputed"]),
    feat_m1,
    feat_c1
)

In [62]:
# 1) load the DataFrame
df = pd.read_csv(csv_path_pairs[-2]["input"])

# 2) build a mask that’s True where values are between 0 and 1
between_0_and_1 = (df >= 0) & (df <= 1)

# 3) check if *all* entries satisfy that condition
if between_0_and_1.values.all():
    print("✅ All values are between 0 and 1")
else:
    # find where the violation occurs
    bad = df[~between_0_and_1]
    print("❌ Some values lie outside [0,1]:")
    print(bad.stack())

❌ Some values lie outside [0,1]:
32   Alvarado_Score                     1.074767
33   Alvarado_Score                     1.001475
63   BMI                                1.070000
130  Stool-is_constipation, diarrhea   -0.000011
239  Stool-is_constipation, diarrhea   -0.000004
358  CRP                               -0.012744
380  CRP                               -0.001279
413  Height                             1.025636
432  CRP                               -0.003478
475  Alvarado_Score                     1.012101
479  Appendix_Diameter                  1.000000
499  Alvarado_Score                     1.064885
508  BMI                                1.045774
536  Alvarado_Score                     1.047445
554  Alvarado_Score                     1.025982
630  Stool-is_constipation, diarrhea   -0.000013
682  Stool-is_constipation, diarrhea   -0.000009
dtype: float64


In [63]:
input_df = pd.read_csv(csv_path_pairs[-2]["input"])
# This will set any value <0 to 0, and any value >1 to 1
input_df.clip(0, 1, inplace=True)

train_masked_vae(
    input_df,
    mask_array = np.hstack([test_apd_mask_num, test_apd_mask_cat])
)

Training on cpu ...
Fold 1: early stopping at epoch 101
Fold 1 final validation loss: 0.4918
Fold 2: early stopping at epoch 100
Fold 2 final validation loss: 0.4962
Fold 3: early stopping at epoch 101
Fold 3 final validation loss: 0.4880
Fold 4: early stopping at epoch 101
Fold 4 final validation loss: 0.4803
Fold 5: early stopping at epoch 101
Fold 5 final validation loss: 0.4708


[0.4917516033132593,
 0.4962394258359095,
 0.4880225274946306,
 0.48029182914277196,
 0.4707906170630119]

##### Apply pre-imputation by BRR to the amputed dataframes

#### 2. <code>the Heart Failure Clinical Records Dataset</code> (no missing values, 299 patients)

##### Load and pre-process the data

In [51]:
csv_path_pairs[2]["ref"]

'/content/drive/My Drive/VAEQL_paper/preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv'

In [52]:
feat_m2, feat_c2 = identify_binary_and_numerical_features(pd.read_csv(csv_path_pairs[2]["ref"]))

In [54]:
test_hf_mask_num, test_hf_mask_cat = generate_masks_for_missingness(
    pd.read_csv(csv_path_pairs[2]["input"]),
    pd.read_csv(csv_path_pairs[2]["ref"]),
    pd.read_csv(csv_path_pairs[2]["amputed"]),
    feat_m2,
    feat_c2
)

In [64]:
input_df = pd.read_csv(csv_path_pairs[2]["input"])
# This will set any value <0 to 0, and any value >1 to 1
input_df.clip(0, 1, inplace=True)

train_masked_vae(
    input_df,
    mask_array = np.hstack([test_hf_mask_num, test_hf_mask_cat])
)

Training on cpu ...
Fold 1: early stopping at epoch 100
Fold 1 final validation loss: 0.6053
Fold 2: early stopping at epoch 100
Fold 2 final validation loss: 0.6061
Fold 3: early stopping at epoch 104
Fold 3 final validation loss: 0.6042
Fold 4: early stopping at epoch 105
Fold 4 final validation loss: 0.6121
Fold 5: early stopping at epoch 102
Fold 5 final validation loss: 0.6315


[0.6052987337112427,
 0.606104830900828,
 0.6042357524236043,
 0.6120965321858723,
 0.6314937765315428]

##### Do the parallel training on GPU and export the VAE models of each dataset

## Defining the MDP Q-learning part of the VAEQL algorithm

In [ ]:
class MDP_Q_learner():
    def init(self,
            device:str,
            input_df: DataFrame,
            states: namedtuple,
            actions: namedtuple,
            gamma: float=0.9, # discount factor
            alpha = 1e-2, # learning rate
            epsilon=0.2,
            ):
        # externally defined input variables, staying constant
        self.DF = input_df
        self.STATES = states
        self.ACTIONS = actions
        self.DEVICE = device
        # variables that stay constantly updated during Q-learning
        self.state = self.calculate_state() # using self.DF
        self.state_next = None
        # on-policy learning, if it doesn't work, might switch to off-policy learning with replay buffer later
        self.policy_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        # alternative choice
        #self.policy_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        # provisional Target-Q table for off-policy learning
        #self.target_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        #self.target_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)

    def calculate_state(self) -> torch.Tensor:
        pass

    def update_state(self) -> None:
        pass

    def calculate_reward(self) -> float:
        pass

    def update_policy_Q_table(self) -> None:
        pass

    def start_new_episode(self, state) -> None:
        self.state = self.calculate_state()
        self.state_next = None
        # in the case of off-policy learning
        #self.target_q = self.policy_q.detach().clone()

    def epsilon_greedy_action(self, state) -> torch.Tensor:
        pass

### Defining the overall data loader, pre-processor and trainer class

# Testing the trainer class above on the five pre-processed datasets